# Assignment: Lightning Pipeline for Chest X-Ray Classification

## Learning goals

In this assignment you will build a complete image classification workflow with **PyTorch Lightning**:

1. Organize data loading in a `LightningDataModule`
2. Encapsulate model logic in a `LightningModule`
3. Train with callbacks and mixed precision
4. Evaluate a classifier on pediatric chest X-rays

You will predict three labels:

| Class | Meaning |
|-------|---------|
| `NORMAL` | No pneumonia |
| `BACTERIAL_PNEUMONIA` | Bacterial infection |
| `VIRAL_PNEUMONIA` | Viral infection |

> **Important:** This notebook is for learning only. A real clinical tool would need rigorous validation, regulatory review, and expert oversight.


## Table of contents

- [Setup (Colab)](#setup)
- [Imports](#imports)
- [Explore the dataset](#explore)
- [Exercise 1 — `XRayDataModule`](#ex1)
- [Exercise 2 — `PneumoniaLitModule`](#ex2)
- [Exercise 3 — Early stopping callback](#ex3)
- [Exercise 4 — Trainer and training loop](#ex4)
- [Full training run](#train)
- [Evaluation](#eval)


<a name="setup"></a>
## Setup (run this first in Colab)

The cell below installs missing packages and downloads course files:

- `xray_viz.py` from the [opencampus course-material repository](https://github.com/opencampus-sh/course-material)
- prepared images and pretrained weights from [opencampus on Hugging Face](https://huggingface.co/datasets/opencampus/chest-xray-pneumonia-3class-balanced)

If you cloned the repository locally and already have the folders/files, you can skip downloads.


In [1]:
import subprocess
import sys
import zipfile
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "lightning", "torchmetrics", "seaborn"],
    check=True,
)

COURSE_RAW = (
    "https://raw.githubusercontent.com/opencampus-sh/course-material/main"
    "/applied-machine-learning/week-06"
)
HF_DATASET = (
    "https://huggingface.co/datasets/opencampus/chest-xray-pneumonia-3class-balanced"
    "/resolve/main"
)

VIZ_FILE = Path("xray_viz.py")
WEIGHTS_FILE = Path("resnet18_chest_xray_classifier_weights.pth")
DATA_MARKER = Path("chest_xray/train/NORMAL")


def wget(url: str, destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["wget", "-q", "-c", "-O", str(destination), url], check=True)


if not VIZ_FILE.exists():
    print("Downloading xray_viz.py from GitHub...")
    wget(f"{COURSE_RAW}/xray_viz.py", VIZ_FILE)

if not DATA_MARKER.exists():
    zip_path = Path("chest_xray_prepared.zip")
    print("Downloading prepared dataset from Hugging Face (~1 GB)...")
    wget(f"{HF_DATASET}/chest_xray_prepared.zip", zip_path)
    with zipfile.ZipFile(zip_path, "r") as archive:
        archive.extractall(".")
    zip_path.unlink(missing_ok=True)

if not WEIGHTS_FILE.exists():
    print("Downloading pretrained weights from Hugging Face...")
    wget(f"{HF_DATASET}/resnet18_chest_xray_classifier_weights.pth", WEIGHTS_FILE)

print("Setup complete.")


Setup complete.


<a name="imports"></a>
## Imports


In [ ]:
import os

import lightning.pytorch as pl
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim

from lightning.pytorch.callbacks import EarlyStopping
from torch.utils.data import DataLoader
from torchmetrics.classification import Accuracy
from torchvision import datasets, models as tv_models, transforms

import xray_viz

torch.set_float32_matmul_precision("medium")


<a name="explore"></a>
## Explore the dataset

We use a prepared three-class version of the public pediatric chest X-ray dataset by Kermany et al. ([Mendeley / DOI 10.17632/rscbjbr9sj.3](https://doi.org/10.17632/rscbjbr9sj.3)). The opencampus release is hosted on Hugging Face and already contains:

- a balanced `train/` split
- a balanced `val/` split
- a small `unseen/` folder for manual spot checks

See `prepare_chest_xray_dataset.py` in this folder if you want to inspect how the archive was built.


In [ ]:
DATA_DIR = "./chest_xray/"


In [ ]:
xray_viz.count_images_by_class(DATA_DIR)


In [ ]:
xray_viz.show_training_samples(DATA_DIR + 'train', seed=42)


## Shared preprocessing

These transforms are used throughout the assignment. Training uses light augmentation; validation only resizes and normalizes.


In [ ]:
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.ToTensor(),
    transforms.Normalize(xray_viz.NORMALIZE_MEAN, xray_viz.NORMALIZE_STD),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(xray_viz.NORMALIZE_MEAN, xray_viz.NORMALIZE_STD),
])


In [ ]:
def make_imagefolder_datasets(train_path, val_path, train_tf, val_tf):
    """Return ImageFolder datasets for training and validation."""
    train_ds = datasets.ImageFolder(train_path, train_tf)
    val_ds = datasets.ImageFolder(val_path, val_tf)
    return train_ds, val_ds


def make_dataloader(train_ds, val_ds, batch_size, for_training):
    """Return either a training or validation DataLoader."""
    dataset = train_ds if for_training else val_ds
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=for_training,
    )


<a name="ex1"></a>
## Exercise 1 — `XRayDataModule`

Implement a Lightning data module that wires together the helpers above.

Fill in the `TODO` sections in:

- `__init__`
- `setup`
- `train_dataloader`
- `val_dataloader`


In [ ]:
class XRayDataModule(pl.LightningDataModule):
    def __init__(self, data_dir, batch_size=64):
        super().__init__()
        # TODO: store data_dir, batch_size, TRAIN_TRANSFORM, and VAL_TRANSFORM
        self.data_dir = ...
        self.batch_size = ...
        self.train_transform = ...
        self.val_transform = ...
        self.train_dataset = None
        self.val_dataset = None

    def setup(self, stage=None):
        train_path = os.path.join(self.data_dir, "train")
        val_path = os.path.join(self.data_dir, "val")
        # TODO: create self.train_dataset and self.val_dataset
        self.train_dataset, self.val_dataset = ...

    def train_dataloader(self):
        # TODO: return the training DataLoader
        return ...

    def val_dataloader(self):
        # TODO: return the validation DataLoader
        return ...


In [ ]:
dm_check = XRayDataModule(data_dir=DATA_DIR, batch_size=8)
dm_check.setup()
train_loader = dm_check.train_dataloader()
val_loader = dm_check.val_dataloader()

print("Training samples:", len(dm_check.train_dataset))
print("Validation samples:", len(dm_check.val_dataset))
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


<a name="ex2"></a>
## Exercise 2 — `PneumoniaLitModule`

Implement the Lightning module that wraps a ResNet-18 backbone.

Provided utilities:


In [ ]:
def build_resnet_backbone(num_classes, weights_path):
    """Load ResNet-18 weights and train only the final linear layer."""
    backbone = tv_models.resnet18(weights=None)
    in_features = backbone.fc.in_features
    backbone.fc = nn.Linear(in_features, num_classes)

    state_dict = torch.load(weights_path, map_location="cpu")
    backbone.load_state_dict(state_dict)

    for parameter in backbone.parameters():
        parameter.requires_grad = False
    for parameter in backbone.fc.parameters():
        parameter.requires_grad = True

    return backbone


def build_optimizer_and_scheduler(model, learning_rate, weight_decay):
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.1, patience=2
    )
    return optimizer, scheduler


In [ ]:
class PneumoniaLitModule(pl.LightningModule):
    def __init__(
        self,
        weights_path,
        num_classes=3,
        learning_rate=1e-3,
        weight_decay=1e-2,
    ):
        super().__init__()
        self.save_hyperparameters()

        # TODO: build self.model with build_resnet_backbone(...)
        # TODO: set self.loss_fn (CrossEntropyLoss) and self.accuracy (Accuracy)
        self.model = ...
        self.loss_fn = ...
        self.accuracy = ...

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx=None):
        # TODO: compute and return training loss
        x, y = ...
        logits = ...
        loss = ...
        return loss

    def validation_step(self, batch, batch_idx=None):
        # TODO: compute loss and accuracy, then log both
        x, y = ...
        logits = ...
        loss = ...
        acc = ...
        self.log_dict({"val_loss": loss, "val_acc": acc}, prog_bar=True)

    def configure_optimizers(self):
        # TODO: use build_optimizer_and_scheduler with hparams
        optimizer, scheduler = ...
        return {
            "optimizer": optimizer,
            "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"},
        }


<a name="ex3"></a>
## Exercise 3 — Early stopping callback

Write `create_early_stopping(num_epochs, accuracy_target)` that returns a Lightning `EarlyStopping` callback which:

- monitors `val_acc`
- stops when `accuracy_target` is reached
- uses patience equal to half of `num_epochs` (integer)
- uses `mode="max"`


In [ ]:
def create_early_stopping(num_epochs, accuracy_target):
    # TODO: instantiate and return EarlyStopping
    return EarlyStopping(
        monitor=...,
        stopping_threshold=...,
        patience=...,
        mode=...,
    )


In [ ]:
callback = create_early_stopping(num_epochs=15, accuracy_target=0.90)
print("monitor:", callback.monitor)
print("threshold:", callback.stopping_threshold)
print("patience:", callback.patience)
print("mode:", callback.mode)


<a name="ex4"></a>
## Exercise 4 — Trainer and training loop

Implement `fit_classifier(model, datamodule, num_epochs, callback, dry_run=False)`.

Requirements:

- create a `pl.Trainer` with `max_epochs=num_epochs`
- use mixed precision (`precision="16-mixed"`)
- pass the callback in a list
- call `trainer.fit(model, datamodule)`
- return `(trainer, model)`


In [ ]:
def fit_classifier(model, datamodule, num_epochs, callback, dry_run=False):
    # TODO: create trainer and run model.fit(...)
    trainer = pl.Trainer(
        max_epochs=...,
        accelerator="auto",
        devices=1,
        precision=...,
        callbacks=...,
        logger=False,
        enable_progress_bar=True,
        enable_model_summary=False,
        enable_checkpointing=False,
        fast_dev_run=dry_run,
    )

    # TODO: start training
    ...

    return trainer, model


<a name="train"></a>
## Full training run

The file `resnet18_chest_xray_classifier_weights.pth` contains weights from a two-stage training run on this dataset (feature extraction, then partial fine-tuning). You will continue from those weights and train the classifier head further.


In [ ]:
BACKBONE_WEIGHTS = "./resnet18_chest_xray_classifier_weights.pth"
BATCH_SIZE = 8
NUM_EPOCHS = 15
ACCURACY_TARGET = 0.80

pl.seed_everything(15)

datamodule = XRayDataModule(DATA_DIR, batch_size=BATCH_SIZE)
model = PneumoniaLitModule(weights_path=BACKBONE_WEIGHTS)
stop_cb = create_early_stopping(NUM_EPOCHS, ACCURACY_TARGET)

trainer, trained_model = fit_classifier(
    model=model,
    datamodule=datamodule,
    num_epochs=NUM_EPOCHS,
    callback=stop_cb,
)

print("Training finished.")


<a name="eval"></a>
## Evaluation

Inspect validation performance, then try a few hold-out images from the `unseen/` folder.


In [ ]:
xray_viz.validation_report(trained_model, datamodule)


In [ ]:
xray_viz.show_validation_predictions(trained_model, datamodule, seed=7)


### Hold-out spot check

These images were excluded from both training and validation:


In [ ]:
HOLDOUT_IMAGE = "./unseen/BACTERIAL_PNEUMONIA/person296_bacteria_1394.jpeg"
xray_viz.predict_and_show(trained_model, datamodule, HOLDOUT_IMAGE)


## Wrap-up

You built a modular Lightning project: data module, model module, callbacks, trainer, and evaluation helpers. That structure scales well when datasets, models, or hardware change.

**Dataset citation:** Kermany, D.; Zhang, K.; Goldbaum, M. *Cell* 2018. DOI [10.17632/rscbjbr9sj.3](https://doi.org/10.17632/rscbjbr9sj.3)
